# b1 · GARCH and GARCH-MIDAS Forecasts

This notebook generates one-month-ahead realised-variance forecasts for XLI, XLB, XLE, XLY and XLF.

1. **GARCH(1,1):** a daily volatility benchmark fitted separately for each ETF.
2. **GARCH-MIDAS:** the same short-run GARCH component plus a macro-driven long-run component. Every parameter, including the MIDAS coefficient `theta`, is estimated separately for each ETF.

Four theory-motivated long-run drivers are estimated with the same protocol: UNRATE (labour market), CPIAUCSL (inflation), DGS3MO (short-term interest rates) and BAA10Y (credit conditions). Their forecast losses are compared below, BAA10Y is chosen as the main specification, and only the GARCH and BAA10Y GARCH-MIDAS forecasts and the BAA10Y `theta` path are exported.

Both models use daily excess returns, a 60-month burn-in, expanding estimation windows, monthly MLE refits and a one-month forecast horizon. The forecast origin is month *t* and the target is month *t + 1*.


## Configuration

Set the input paths, ETF universe, estimation settings, and fit cache. The cache is keyed to the data the fits use (the five ETFs' excess returns and the four macro drivers), so saved estimates are reused only when those data are unchanged.

In [1]:
import sys
from pathlib import Path

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))
from utils.paths import PROJECT_ROOT, MODELS_DIR, DATASET_DIR, FORECAST_DIR, FORECAST_RESULTS_DIR

import os, json, hashlib
import numpy as np
import pandas as pd
from scipy.optimize import minimize


PRICE_PATH = DATASET_DIR / "price" / "clean" / "daily_returns_panel_ver2.csv"
MACRO_PATH = DATASET_DIR / "macro_variables" / "final_clean" / "final_macro_data.csv"

# Hand-off files for the separate evaluation notebook.
OUT_DIR = os.path.join(PROJECT_ROOT, "models", "forecasting", "forecasts")
OUT_TIDY = os.path.join(OUT_DIR, "garch_forecasts.csv")
OUT_THETA = os.path.join(OUT_DIR, "garchmidas_theta_by_origin.csv")
# Saved fits make an interrupted or repeated run resumable; without them every fit is re-estimated.
CACHE_DIR = os.path.join(PROJECT_ROOT, "supporting", "saved_fits")
USE_FIT_CACHE = os.environ.get("DSE4101_USE_FIT_CACHE", "1") == "1"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

TICKERS = ["XLI", "XLB", "XLE", "XLY", "XLF"]
SAMPLE_START = "2003-02"
BURN_IN = 60
REFIT_EVERY = 1
HORIZON = 1
K_MIDAS = 1
N_STARTS = 3
N_RANDOM = 10
# Candidate long-run drivers; BAA10Y is the main specification (see the driver comparison below).
DRIVERS = ["UNRATE", "CPIAUCSL", "DGS3MO", "BAA10Y"]
DRIVER = "BAA10Y"
MIDAS_LABEL = "garch_midas_" + DRIVER.lower()
MODELS = ["garch", MIDAS_LABEL]

print(f"Configuration ready: {len(TICKERS)} ETFs; GARCH-MIDAS drivers = {', '.join(DRIVERS)}; main = {DRIVER}.")
print(f"Fit cache enabled: {USE_FIT_CACHE}; cache directory: {CACHE_DIR}")


Configuration ready: 5 ETFs; GARCH-MIDAS drivers = UNRATE, CPIAUCSL, DGS3MO, BAA10Y; main = BAA10Y.
Fit cache enabled: True; cache directory: ./supporting/saved_fits


## Data preparation

Load daily excess returns and the four monthly macro drivers, align them to a common monthly sample, and calculate realised variance as the monthly sum of squared daily excess returns.

BAA10Y and DGS3MO are market data observed by the end of the forecast-origin month. UNRATE and CPIAUCSL for month *t* are the values released during month *t* (reference month *t − 1*), so they are also known at the origin. With `K_MIDAS = 1`, the forecast for month *t + 1* uses each driver's value for month *t* only.


In [2]:
# ---- Daily excess-return panel ----
px = pd.read_csv(PRICE_PATH, index_col="date", parse_dates=True)
required_return_cols = [f"xret_{tk}" for tk in TICKERS]
missing_returns = [c for c in required_return_cols if c not in px.columns]
assert not missing_returns, f"Missing return columns: {missing_returns}"
assert px.index.is_monotonic_increasing and not px.index.has_duplicates, "Return dates must be sorted and unique"

xr = px[required_return_cols].rename(columns=lambda c: c[len("xret_"):])
assert not xr.isna().any().any(), "Daily excess returns contain missing values"
xr = xr[xr.index >= pd.Period(SAMPLE_START, "M").start_time]  # January 2003 has one session

# ---- Monthly macro panel ----
macro_all = pd.read_csv(MACRO_PATH, index_col="date", parse_dates=True)
macro_all.index = macro_all.index.to_period("M")
missing_drivers = [d for d in DRIVERS if d not in macro_all.columns]
assert not missing_drivers, f"Macro file is missing {missing_drivers}"
assert macro_all.index.is_monotonic_increasing and not macro_all.index.has_duplicates, "Macro dates must be sorted and unique"
assert len(macro_all.index) > 1 and set(np.diff(macro_all.index.asi8)) == {1}, "Macro panel has calendar gaps"

# ---- Common monthly grid ----
GRID = pd.period_range(max(pd.Period(SAMPLE_START, "M"), macro_all.index[0], xr.index[0].to_period("M")),
                       min(macro_all.index[-1], xr.index[-1].to_period("M")), freq="M")
N_MONTHS = len(GRID)
macro_matrix = macro_all.reindex(GRID)
assert macro_matrix[DRIVERS].notna().all().all(), "A macro driver has missing values inside the common grid"

# No partial monthly return observations are allowed inside the estimation/forecast grid.
sessions = pd.Series(1, index=xr.index).groupby(xr.index.to_period("M")).sum().reindex(GRID)
weekdays = pd.Series({p: np.busday_count(p.start_time.date(), (p + 1).start_time.date()) for p in GRID})
partial_months = list(GRID[(sessions < 0.6 * weekdays).to_numpy()])
assert not partial_months, f"Partial months inside the grid: {partial_months}"

# Target: monthly realised variance = sum of squared daily excess returns.
rv2 = xr.pow(2).resample("ME").sum()
rv2.index = rv2.index.to_period("M")
rv2 = rv2.reindex(GRID)
assert not rv2.isna().any().any() and (rv2 > 0).all().all(), "Realised variance must be finite and positive"


# Saved fits are keyed to the text of the columns the fits use (the five ETFs' excess returns and the
# four drivers), so changes to other columns, such as WTI, or to line endings do not invalidate them.
def _fingerprint(path, cols):
    raw = pd.read_csv(path, usecols=["date"] + cols, dtype=str)[["date"] + cols]
    text = "\n".join(",".join(map(str, row)) for row in raw.itertuples(index=False))
    return hashlib.md5(text.encode()).hexdigest()[:12]


DATA_FINGERPRINT = _fingerprint(PRICE_PATH, required_return_cols) + "_" + _fingerprint(MACRO_PATH, DRIVERS)
print(f"Data ready: {GRID[0]} to {GRID[-1]} ({N_MONTHS} months; {len(xr)} return days).")
print("[OK] sorted/unique dates, complete macro grid, no partial return months, finite positive RV")


Data ready: 2003-02 to 2025-12 (275 months; 5766 return days).
[OK] sorted/unique dates, complete macro grid, no partial return months, finite positive RV


## Forecast design

Prepare the daily return arrays used by the likelihood, estimate the next month’s trading sessions from past calendar patterns, and define the expanding-window forecast calendar. The first forecast is made after a 60-month burn-in, and each forecast targets the following month. The monthly forecast is the expected sum of the daily conditional variances over the target month's estimated trading sessions, plus N·μ̂²; it uses only returns and macro data up to the origin month.

In [3]:
def build_daily():
    """Concatenate the five tickers once; each fit takes index slices of these arrays."""
    pos = {p: i for i, p in enumerate(GRID)}
    rets_l, mids_l, blo, bhi = [], [], [], []
    off = 0
    for tk in TICKERS:
        s = xr[tk]
        s = s[[p in pos for p in s.index.to_period("M")]]
        rets_l.append(s.to_numpy(dtype=np.float64))
        mids_l.append(np.array([pos[p] for p in s.index.to_period("M")], dtype=np.int64))
        blo.append(off); off += len(s); bhi.append(off)
    return (np.ascontiguousarray(np.concatenate(rets_l)),
            np.ascontiguousarray(np.concatenate(mids_l)),
            np.array(blo, dtype=np.int64), np.array(bhi, dtype=np.int64))


RETS, MIDS, BLO, BHI = build_daily()
# ---- session counts, built ex ante ----
OBS = np.zeros((len(TICKERS), N_MONTHS), dtype=np.int64)
for _k in range(len(TICKERS)):
    np.add.at(OBS[_k], MIDS[BLO[_k]:BHI[_k]], 1)
WD = np.array([np.busday_count(p.start_time.date(), (p + 1).start_time.date())
               for p in GRID], dtype=np.int64)
MOY = np.array([p.month for p in GRID], dtype=np.int64)


def estimate_sessions(target_idx, obs_k, min_share=0.6):
    """Sessions in month `target_idx`, using ONLY months strictly before it: weekday count
    (pure calendar) minus the average holiday shortfall in earlier months of the same
    calendar month. The target month's own returns are never touched."""
    if target_idx <= 0:
        return int(WD[target_idx])
    o, w, m = obs_k[:target_idx], WD[:target_idx], MOY[:target_idx]
    keep = o >= min_share * w
    if not np.any(keep):
        return int(WD[target_idx])
    short = (w - o)[keep]
    same = short[m[keep] == MOY[target_idx]]
    sf = float(np.mean(same)) if len(same) else float(np.mean(short))
    return int(min(max(int(round(WD[target_idx] - sf)), 1), int(WD[target_idx])))


def slice_bounds(K, n_months_obs):
    """Daily slice [lo, hi) per ticker covering months K .. n_months_obs-1. Starting at
    month K guarantees every month the filter sees has a complete MIDAS lookback."""
    lo = np.empty(len(TICKERS), dtype=np.int64)
    hi = np.empty(len(TICKERS), dtype=np.int64)
    for k in range(len(TICKERS)):
        seg = MIDS[BLO[k]:BHI[k]]
        lo[k] = BLO[k] + int(np.searchsorted(seg, K, side="left"))
        hi[k] = BLO[k] + int(np.searchsorted(seg, n_months_obs, side="left"))
    return lo, hi


# ---- the protocol dates, derived rather than typed ----
FIRST_ORIGIN = K_MIDAS + BURN_IN - 1          # the first window, months K .. FIRST_ORIGIN, is the burn-in
LAST_ORIGIN = N_MONTHS - 2                    # its target is the last grid month
ORIGINS = list(range(FIRST_ORIGIN, LAST_ORIGIN + 1))
TARGETS = pd.PeriodIndex([GRID[o + HORIZON] for o in ORIGINS])
print(f"Forecast design ready: {len(TARGETS)} monthly forecasts per ETF, from {TARGETS[0]} to {TARGETS[-1]}.")


Forecast design ready: 214 monthly forecasts per ETF, from 2008-03 to 2025-12.


## Model definition

Define the shared daily GARCH recursion and the GARCH-MIDAS extension. GARCH sets the macro effect to zero; GARCH-MIDAS estimates the macro long-run component separately for each ETF. 

In [4]:
try:
    from numba import njit
    print("Numba acceleration enabled.")
except Exception as exc:
    print(f"WARNING: Numba is unavailable ({exc}); GARCH fits will use slow Python loops.")
    def njit(*a, **k):
        def deco(f):
            return f
        return deco if not a else a[0]

LOG2PI = float(np.log(2.0 * np.pi))
MAX_PERSIST = 0.9995
LOG_TAU_MIN, LOG_TAU_MAX = -30.0, 5.0
Z_CLAMP = 25.0
OMEGA2_MIN, OMEGA2_MAX = 1.001, 50.0
OMEGA2_FIXED_K1 = 2.0          # at K=1 omega2 has no effect; it is reported as 2.0


@njit(cache=True)
def _sig(x):
    if x > Z_CLAMP:
        x = Z_CLAMP
    elif x < -Z_CLAMP:
        x = -Z_CLAMP
    return 1.0 / (1.0 + np.exp(-x))


def _logit(p):
    p = min(max(float(p), 1e-9), 1 - 1e-9)
    return float(np.log(p / (1 - p)))


@njit(cache=True)
def midas_weights(K, omega2):
    """Beta weights w_j propto (1 - j/(K+1))^(omega2-1), j = 1..K, normalised to sum 1.

    At K = 1 this is [1.0] for every omega2: the shape parameter has no effect.
    """
    w = np.empty(K)
    for j in range(1, K + 1):
        base = 1.0 - j / (K + 1.0)
        if base <= 0.0:
            base = 1e-12
        w[j - 1] = base ** (omega2 - 1.0)
    s = w.sum()
    if s <= 0.0 or not np.isfinite(s):
        for j in range(K):
            w[j] = 1.0 / K
        return w
    return w / s


@njit(cache=True)
def midas_index(x, K, omega2, n_months):
    """mid[t] = sum_{j=1..K} w_j * x[t-j]. Defined (and used) only for t >= K."""
    w = midas_weights(K, omega2)
    out = np.zeros(n_months)
    for t in range(K, n_months):
        s = 0.0
        for j in range(1, K + 1):
            s += w[j - 1] * x[t - j]
        out[t] = s
    return out


@njit(cache=True)
def filter_core(rets, mids, tau, mu, alpha, beta, lo, hi):
    """GARCH recursion over daily slice [lo, hi). Returns
    (loglik, n_used, g_last, resid_last_sq, ok).

    g_last is the g of the final observed day; resid_last_sq is (r_last - mu)^2. Together
    they are exactly what the one-step forecast needs.
    """
    n = hi - lo
    if n < 2:
        return 0.0, 0, 1.0, 0.0, False
    g = 1.0
    ll = 0.0
    used = 0
    resid_prev_sq = 0.0
    for i in range(lo, hi):
        tau_t = tau[mids[i]]
        if tau_t != tau_t or tau_t <= 0.0:
            return 0.0, 0, 1.0, 0.0, False
        if i > lo:
            g = (1.0 - alpha - beta) + alpha * resid_prev_sq / tau_t + beta * g
            if g < 1e-10:
                g = 1e-10
            elif g > 1e10:
                g = 1e10
        r = rets[i] - mu
        resid_prev_sq = r * r
        v = g * tau_t
        if v < 1e-16:
            v = 1e-16
        ll += -0.5 * (LOG2PI + np.log(v) + r * r / v)
        used += 1
    return ll, used, g, resid_prev_sq, True


@njit(cache=True)
def nll_total(z, rets, mids, starts, ends, xs, K, n_months, n_tick, use_theta):
    theta = z[4 * n_tick] if use_theta == 1 else 0.0
    if use_theta == 1:
        omega2 = OMEGA2_MIN + (OMEGA2_MAX - OMEGA2_MIN) * _sig(z[4 * n_tick + 1])
        mid = midas_index(xs, K, omega2, n_months)
    else:
        mid = np.zeros(n_months)

    total = 0.0
    for k in range(n_tick):
        mu = z[k]
        persist = MAX_PERSIST * _sig(z[n_tick + k])
        split = _sig(z[2 * n_tick + k])
        alpha = persist * split
        beta = persist * (1.0 - split)
        m = z[3 * n_tick + k]
        tau = np.empty(n_months)
        for t in range(n_months):
            lt = m + theta * mid[t]
            if lt < LOG_TAU_MIN:
                lt = LOG_TAU_MIN
            elif lt > LOG_TAU_MAX:
                lt = LOG_TAU_MAX
            tau[t] = np.exp(lt)
        ll, used, _, _, ok = filter_core(rets, mids, tau, mu, alpha, beta, starts[k], ends[k])
        if (not ok) or used == 0 or ll != ll:
            return 1e12
        total += ll
    if total != total:
        return 1e12
    return -total


@njit(cache=True)
def expected_g_sum(g1, rho, n_days):
    """E[sum_{k=1..n} g_k] given a deterministic g_1, since E[g_k] = 1 + rho^{k-1}(g_1 - 1)."""
    if n_days <= 0:
        return 0.0
    if abs(1.0 - rho) < 1e-12:
        return n_days * g1
    return n_days + (g1 - 1.0) * (1.0 - rho ** n_days) / (1.0 - rho)


@njit(cache=True)
def nll_grad(z, rets, mids, starts, ends, xs, K, n_months, n_tick, use_theta):
    """Central-difference gradient of nll_total. Accurate to O(h^2), far more precise than
    the one-sided default, which is what lets the optimiser reach the exact maximum."""
    gr = np.zeros(z.shape[0])
    for i in range(z.shape[0]):
        h = 1e-5 * max(1.0, abs(z[i]))
        zp = z.copy()
        zp[i] += h
        zm = z.copy()
        zm[i] -= h
        gr[i] = (nll_total(zp, rets, mids, starts, ends, xs, K, n_months, n_tick, use_theta)
                 - nll_total(zm, rets, mids, starts, ends, xs, K, n_months, n_tick, use_theta)) / (2.0 * h)
    return gr

print("Model core ready.")


Model core ready.


In [5]:
def polish(z0, args):
    """Climb from z0 to the exact top of the likelihood.

    Each parameter is first rescaled by its own curvature (diagonal preconditioning), because
    mu is about a thousand times more sensitive than the other parameters; without this the
    optimiser crawls. It then runs L-BFGS-B with the accurate gradient and tight tolerances.
    Returns (z, negative log-likelihood).
    """
    f0 = nll_total(z0, *args)
    s = np.ones_like(z0)
    for i in range(len(z0)):
        h = 1e-4 * max(1.0, abs(z0[i]))
        zp = z0.copy(); zp[i] += h
        zm = z0.copy(); zm[i] -= h
        curv = (nll_total(zp, *args) - 2.0 * f0 + nll_total(zm, *args)) / h ** 2
        if curv > 1e-8:
            s[i] = 1.0 / np.sqrt(curv)
    r = minimize(lambda u: nll_total(z0 + s * u, *args), np.zeros_like(z0),
                 jac=lambda u: s * nll_grad(z0 + s * u, *args), method="L-BFGS-B",
                 options=dict(maxiter=20000, maxfun=400000, ftol=1e-15, gtol=1e-9))
    if np.isfinite(r.fun) and r.fun <= f0:
        return z0 + s * r.x, float(r.fun)
    return z0, float(f0)


class GarchMidas:
    """GARCH-MIDAS for the ETFs passed in. GARCH-MIDAS is always fitted on one ETF at a time
    (fit_midas_etf), so theta is ETF-specific.

    use_theta=False pins theta to 0, making tau a per-ticker constant and recovering
    GARCH(1,1) exactly. Both models share this class, likelihood, optimiser and forecast,
    so they cannot differ through an implementation detail rather than an economic one.
    """

    def __init__(self, K, tickers, use_theta=True, n_starts=3, seed=0, n_random=0):
        self.K = int(K)
        self.tickers = list(tickers)
        self.n_tick = len(self.tickers)
        self.use_theta = bool(use_theta)
        self.n_starts = int(n_starts)
        self.seed = int(seed)
        self.n_random = int(n_random)
        self.params = None
        self.x_mean, self.x_std = 0.0, 1.0
        self.fit_ok, self.loglik = False, np.nan
        self.conv_gap = np.nan

    def _standardise(self, x_raw, freeze=False):
        x = np.asarray(x_raw, dtype=np.float64)
        if freeze:
            sd = float(np.std(x))
            self.x_mean = float(np.mean(x))
            self.x_std = sd if (np.isfinite(sd) and sd > 1e-12) else 1.0
        return (x - self.x_mean) / self.x_std

    def fit(self, rets, mids, starts, ends, macro_raw, n_months, extra_z0=None):
        """macro_raw must cover months 0..n_months-1; only these months are used.

        extra_z0: optional warm-start vectors (here, the ETF's GARCH solution with
        theta = 0). They are only starting points for the optimiser;
        they carry no data, so they cannot leak information."""
        if len(macro_raw) < n_months:
            raise ValueError(f"macro_raw has {len(macro_raw)} months, need {n_months}")
        for k in range(self.n_tick):
            seg = mids[starts[k]:ends[k]]
            if len(seg) and int(seg.max()) >= n_months:
                raise ValueError("daily data reaches the target month (leak guard)")

        self._standardise(macro_raw[:n_months], freeze=True)
        xs = self._standardise(macro_raw[:n_months])
        ut = 1 if self.use_theta else 0
        args = (rets, mids, starts, ends, xs, self.K, int(n_months), self.n_tick, ut)

        base_mu = np.array([float(np.mean(rets[starts[k]:ends[k]])) for k in range(self.n_tick)])
        base_m = np.array([float(np.log(max(np.var(rets[starts[k]:ends[k]]), 1e-12)))
                           for k in range(self.n_tick)])
        grid = [(0.06, 0.92, 0.0, 3.0), (0.10, 0.85, -0.25, 6.0), (0.03, 0.95, 0.25, 1.5)]
        z0s = []
        for (a, b, th, w2) in grid[:max(1, self.n_starts)]:
            z0s.append(np.concatenate([
                base_mu,
                np.full(self.n_tick, _logit((a + b) / MAX_PERSIST)),
                np.full(self.n_tick, _logit(a / (a + b))),
                base_m,
                [th, _logit((w2 - OMEGA2_MIN) / (OMEGA2_MAX - OMEGA2_MIN))]]))
        if extra_z0:
            z0s.extend(np.asarray(z, dtype=np.float64) for z in extra_z0)
        # Random starting points, spread over the plausible range. The likelihood can have
        # several peaks, and starting in many different places makes it very likely that one
        # of the searches reaches the highest. The seed is fixed, so every run is reproducible.
        # Starting points contain no data, so they cannot leak information.
        rng_s = np.random.default_rng(self.seed)
        for _ in range(self.n_random):
            a = rng_s.uniform(0.02, 0.15, self.n_tick)
            b = rng_s.uniform(0.80, 0.985 - a)
            z0s.append(np.concatenate([
                base_mu,
                [_logit(v) for v in (a + b) / MAX_PERSIST],
                [_logit(v) for v in a / (a + b)],
                base_m + rng_s.uniform(-1.0, 1.0, self.n_tick),
                [float(rng_s.uniform(-0.6, 0.6)) if self.use_theta else 0.0,
                 _logit((OMEGA2_FIXED_K1 - OMEGA2_MIN) / (OMEGA2_MAX - OMEGA2_MIN))]]))

        res = [minimize(nll_total, z0, args=args, method="L-BFGS-B",
                        options=dict(maxiter=800)) for z0 in z0s]
        good = [r for r in res if r.success and np.isfinite(r.fun) and r.fun < 1e11]
        if not good:
            rng = np.random.default_rng(self.seed)
            for _ in range(4):
                a = float(rng.uniform(0.03, 0.14))
                b = float(rng.uniform(0.75, 0.95 - a))
                z0 = np.concatenate([
                    base_mu, np.full(self.n_tick, _logit((a + b) / MAX_PERSIST)),
                    np.full(self.n_tick, _logit(a / (a + b))), base_m,
                    [float(rng.uniform(-0.4, 0.4)),
                     _logit((float(rng.uniform(1.3, 12.0)) - OMEGA2_MIN)
                            / (OMEGA2_MAX - OMEGA2_MIN))]])
                r = minimize(nll_total, z0, args=args, method="L-BFGS-B",
                             options=dict(maxiter=800))
                res.append(r)
                if r.success and np.isfinite(r.fun) and r.fun < 1e11:
                    good.append(r)
                    break

        # Maximum likelihood = the lowest negative log-likelihood among ALL finite runs.
        # L-BFGS-B can report success=False after a line-search stop at a point that is
        # still the best one found; discarding it would throw away the actual maximum.
        pool = [r for r in res if np.isfinite(r.fun) and r.fun < 1e11]
        if not pool:
            self.fit_ok, self.params, self.loglik = False, None, np.nan
            return self
        # Climb EVERY candidate to its exact top, then keep the highest. The likelihood can
        # have more than one peak, so stopping any candidate early could pick the wrong one.
        tops = [polish(r.x, args) for r in pool]
        z, f_best = min(tops, key=lambda t: t[1])
        # Convergence certificate: a second climb from the winner must find nothing more.
        z2, f2 = polish(z, args)
        self.conv_gap = float(max(f_best - f2, 0.0))
        if f2 < f_best:
            z, f_best = z2, f2
        self.fit_ok = bool(np.isfinite(f_best))
        n = self.n_tick
        per = {}
        for k, tk in enumerate(self.tickers):
            persist = MAX_PERSIST * _sig(z[n + k])
            split = _sig(z[2 * n + k])
            per[tk] = dict(mu=float(z[k]), alpha=float(persist * split),
                           beta=float(persist * (1 - split)), m=float(z[3 * n + k]))
        om = (float(OMEGA2_MIN + (OMEGA2_MAX - OMEGA2_MIN) * _sig(z[4 * n + 1]))
              if (self.use_theta and self.K > 1) else float(OMEGA2_FIXED_K1))
        self.params = dict(per_ticker=per,
                           theta=float(z[4 * n]) if self.use_theta else 0.0,
                           omega2=om, x_mean=self.x_mean, x_std=self.x_std,
                           K=self.K, use_theta=self.use_theta)
        self.loglik = float(-f_best)
        return self

    def set_params(self, p):
        self.params = dict(p)
        self.params["per_ticker"] = {k: dict(v) for k, v in p["per_ticker"].items()}
        self.x_mean, self.x_std = p["x_mean"], p["x_std"]
        self.K = int(p["K"])
        self.use_theta = bool(p["use_theta"])
        self.fit_ok = True
        return self

    def params_to_z(self, p, theta=None):
        """Map a parameter dict back to the optimiser's unconstrained vector (warm start)."""
        n = self.n_tick
        z = np.empty(4 * n + 2)
        for k, tk in enumerate(self.tickers):
            t = p["per_ticker"][tk]
            persist = t["alpha"] + t["beta"]
            z[k] = t["mu"]
            z[n + k] = _logit(persist / MAX_PERSIST)
            z[2 * n + k] = _logit(t["alpha"] / persist)
            z[3 * n + k] = t["m"]
        z[4 * n] = p["theta"] if theta is None else theta
        z[4 * n + 1] = _logit((p.get("omega2", OMEGA2_FIXED_K1) - OMEGA2_MIN) / (OMEGA2_MAX - OMEGA2_MIN))
        return z

    def forecast_h1(self, ticker, rets, mids, lo, hi, macro_raw, origin_idx, n_days_target):
        """E[sum of squared daily returns] in month origin_idx+1, using only data through
        month origin_idx.

        At h = 1 the target month's MIDAS sum reaches months origin_idx down to
        origin_idx+1-K, every one of which is observable at the origin, so no truncation
        or weight renormalisation is required.
        """
        p = self.params
        if p is None:
            return np.nan
        tp = p["per_ticker"][ticker]
        target_idx = origin_idx + 1
        if hi > lo and int(mids[lo:hi].max()) > origin_idx:
            raise ValueError("training days reach past the origin month (leak guard)")
        if len(macro_raw) < origin_idx + 1:
            raise ValueError("macro array does not cover the origin month")
        if target_idx < self.K or n_days_target <= 0:
            return np.nan

        n_obs = origin_idx + 1
        xs = self._standardise(macro_raw[:n_obs])
        if self.use_theta:
            w = midas_weights(self.K, p["omega2"])
            mid_t = 0.0
            for j in range(1, self.K + 1):
                idx = target_idx - j
                if idx < 0 or idx > origin_idx:
                    raise ValueError("MIDAS lag reaches an unobservable month (leak guard)")
                mid_t += w[j - 1] * xs[idx]
            mid_in = midas_index(xs, self.K, p["omega2"], n_obs)
        else:
            mid_t = 0.0
            mid_in = np.zeros(n_obs)

        tau_target = float(np.exp(np.clip(tp["m"] + p["theta"] * mid_t,
                                          LOG_TAU_MIN, LOG_TAU_MAX)))
        if not np.isfinite(tau_target) or tau_target <= 0:
            return np.nan

        tau_in = np.exp(np.clip(tp["m"] + p["theta"] * mid_in, LOG_TAU_MIN, LOG_TAU_MAX))
        _, _, g_last, resid_last_sq, ok = filter_core(
            rets, mids, tau_in, tp["mu"], tp["alpha"], tp["beta"], lo, hi)
        if not ok:
            return np.nan

        rho = tp["alpha"] + tp["beta"]
        # first day of the target month is deterministic: it uses the last OBSERVED return,
        # scaled by tau of the target month (the within-month convention of Engle, Ghysels and Sohn, 2013)
        g1 = (1.0 - rho) + tp["alpha"] * resid_last_sq / tau_target + tp["beta"] * g_last
        s = expected_g_sum(g1, rho, int(n_days_target))
        return float(tau_target * s + n_days_target * tp["mu"] ** 2)

print("Model estimator ready.")

Model estimator ready.


## Expanding-window backtest

Define the expanding-window fitting paths, then refit GARCH and the four GARCH-MIDAS candidates monthly using only information available at each forecast origin. GARCH-MIDAS uses that month’s ETF-specific GARCH estimate as one optimiser starting value.

Existing fits are reused from `supporting/saved_fits` when the data fingerprint and modelling settings are unchanged. This avoids re-estimating unchanged monthly MLE fits on a subsequent run.


In [6]:
def _cache_path(tag):
    return os.path.join(CACHE_DIR, hashlib.md5(tag.encode()).hexdigest() + ".json")


def cache_get(tag):
    if not USE_FIT_CACHE:
        return None
    p = _cache_path(tag)
    if not os.path.exists(p):
        return None
    with open(p) as f:
        obj = json.load(f)
    return obj if obj.get("tag") == tag else None      # guard against hash collisions


def cache_put(tag, obj):
    if not USE_FIT_CACHE:
        return
    obj = dict(obj, tag=tag)
    tmp = _cache_path(tag) + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f)
    os.replace(tmp, _cache_path(tag))          # an interrupted write leaves no partial file


def spec_for(label, use_theta):
    return (f"phase2|{label}|theta={use_theta}|K={K_MIDAS}|burn={BURN_IN}|refit={REFIT_EVERY}"
            f"|grid={GRID[0]}..{GRID[-1]}|{TICKERS}|starts={N_STARTS}+{N_RANDOM}rand+nested+polish-all"
            f"|data={DATA_FINGERPRINT}")


def midas_label(driver):
    return "garch_midas_" + driver.lower()


def midas_tag(driver, tk, origin):
    return f"{spec_for(midas_label(driver) + '_sector_theta', True)}|etf={tk}|origin={origin}"


def fit_record(m):
    """What is saved in the cache for one fit."""
    return dict(params=m.params, fit_ok=bool(m.fit_ok),
                loglik=float(m.loglik) if np.isfinite(m.loglik) else None, conv_gap=float(m.conv_gap))


def fit_garch(origin):
    """The GARCH(1,1) fit at one forecast origin.

    GARCH has no parameter shared across ETFs, so its likelihood is a sum of five independent
    ETF terms and the joint fit returns each ETF's own maximum-likelihood estimates. Fits on
    daily excess returns of months K..origin. Starting points: N_STARTS fixed ones and N_RANDOM
    random ones (seeded by the origin month).
    """
    lo, hi = slice_bounds(K_MIDAS, origin + 1)
    m = GarchMidas(K_MIDAS, TICKERS, use_theta=False, n_starts=N_STARTS, seed=100_000 + origin,
                   n_random=N_RANDOM)
    return m.fit(RETS, MIDS, lo, hi, np.zeros(origin + 1), origin + 1)


def fit_midas_etf(tk, origin, macro_np, garch_params):
    """The GARCH-MIDAS fit for one ETF at one forecast origin.

    Every parameter, theta included, is specific to this ETF. Starting points: N_STARTS fixed
    ones, N_RANDOM random ones (seeded by ETF and origin), and this ETF's GARCH(1,1) solution
    with theta = 0. Because GARCH-MIDAS nests GARCH, that start guarantees the ETF's GARCH-MIDAS
    fit is at least as good as its GARCH fit.
    """
    k = TICKERS.index(tk)
    lo, hi = slice_bounds(K_MIDAS, origin + 1)
    m = GarchMidas(K_MIDAS, [tk], use_theta=True, n_starts=N_STARTS,
                   seed=100_000 + 1_000 * (k + 1) + origin, n_random=N_RANDOM)
    nested = dict(per_ticker={tk: garch_params["per_ticker"][tk]}, theta=0.0, omega2=garch_params["omega2"])
    return m.fit(RETS, MIDS, lo[k:k + 1].copy(), hi[k:k + 1].copy(), macro_np[:origin + 1], origin + 1,
                 extra_z0=[m.params_to_z(nested, theta=0.0)])


def etf_garch_loglik(tk, origin, garch_params):
    """One ETF's term of the GARCH log-likelihood at the given parameters."""
    k = TICKERS.index(tk)
    lo, hi = slice_bounds(K_MIDAS, origin + 1)
    g = GarchMidas(K_MIDAS, [tk], use_theta=False)
    z = g.params_to_z(dict(per_ticker={tk: garch_params["per_ticker"][tk]}, theta=0.0,
                           omega2=garch_params["omega2"]))
    return -nll_total(z, RETS, MIDS, lo[k:k + 1].copy(), hi[k:k + 1].copy(), np.zeros(origin + 1),
                      K_MIDAS, origin + 1, 1, 0)


def walk_forward_garch():
    """GARCH(1,1): at each origin T, fit on months K..T and forecast month T+1."""
    spec = spec_for("garch", False)
    rows, refits, params = [], [], {}
    for origin in ORIGINS:
        tag = f"{spec}|origin={origin}"
        hit = cache_get(tag)
        if hit is None:
            hit = fit_record(fit_garch(origin))
            cache_put(tag, hit)
        assert hit["fit_ok"], f"GARCH fit failed at {GRID[origin]}"
        params[origin] = hit["params"]
        refits.append(dict(origin=str(GRID[origin]), fit_ok=hit["fit_ok"], conv_gap=hit["conv_gap"]))
        model = GarchMidas(K_MIDAS, TICKERS, use_theta=False).set_params(hit["params"])
        lo, hi = slice_bounds(K_MIDAS, origin + 1)
        for k, tk in enumerate(TICKERS):
            nd = estimate_sessions(origin + 1, OBS[k])
            fc = model.forecast_h1(tk, RETS, MIDS, lo[k], hi[k], np.zeros(origin + 1), origin, nd)
            rows.append(dict(model="garch", etf=tk, origin=GRID[origin], target=GRID[origin + 1], rv2_hat=fc))
    return pd.DataFrame(rows), pd.DataFrame(refits), params


def walk_forward_midas(driver, garch_par):
    """GARCH-MIDAS for each ETF separately: at each origin T, fit on months K..T and forecast month T+1."""
    macro_np = macro_matrix[driver].to_numpy(float)
    rows, refits = [], []
    for k, tk in enumerate(TICKERS):
        for origin in ORIGINS:
            tag = midas_tag(driver, tk, origin)
            hit = cache_get(tag)
            if hit is None:
                hit = fit_record(fit_midas_etf(tk, origin, macro_np, garch_par[origin]))
                cache_put(tag, hit)
            assert hit["fit_ok"], f"GARCH-MIDAS ({driver}) fit failed for {tk} at {GRID[origin]}"
            p = hit["params"]
            refits.append(dict(etf=tk, origin=str(GRID[origin]), fit_ok=hit["fit_ok"], conv_gap=hit["conv_gap"],
                               loglik=hit["loglik"], garch_loglik=etf_garch_loglik(tk, origin, garch_par[origin])))
            model = GarchMidas(K_MIDAS, [tk], use_theta=True).set_params(p)
            lo, hi = slice_bounds(K_MIDAS, origin + 1)
            nd = estimate_sessions(origin + 1, OBS[k])
            fc = model.forecast_h1(tk, RETS, MIDS, lo[k], hi[k], macro_np[:origin + 1], origin, nd)
            rows.append(dict(model=midas_label(driver), etf=tk, origin=GRID[origin], target=GRID[origin + 1], rv2_hat=fc,
                             theta=p["theta"], params_from=str(GRID[origin])))
    return pd.DataFrame(rows), pd.DataFrame(refits)


print("Backtest fitting routines ready.")

Backtest fitting routines ready.


In [7]:
FC, REF = {}, {}
FC["garch"], REF["garch"], GARCH_PAR = walk_forward_garch()
for drv in DRIVERS:
    FC[midas_label(drv)], REF[midas_label(drv)] = walk_forward_midas(drv, GARCH_PAR)

# Every GARCH-MIDAS fit must improve or match its nested GARCH likelihood, and all fits
# must satisfy the numerical convergence certificate.
for drv in DRIVERS:
    r = REF[midas_label(drv)]
    assert (r.loglik - r.garch_loglik).min() >= -1e-6, f"{drv}: a fit is below its nested GARCH likelihood"
assert max(r.conv_gap.max() for r in REF.values()) < 1e-6
for label, forecasts in FC.items():
    assert len(forecasts) == len(TICKERS) * len(TARGETS), f"{label}: unexpected forecast count"
    assert np.isfinite(forecasts.rv2_hat).all() and (forecasts.rv2_hat > 0).all(), f"{label}: invalid forecast"

print(f"Backtest complete: {len(TARGETS)} forecasts per ETF for GARCH and each of {len(DRIVERS)} GARCH-MIDAS drivers.")
print(f"{sum((r.conv_gap < 1e-6).sum() for r in REF.values())}/{sum(len(r) for r in REF.values())} fits passed the convergence check.")


Backtest complete: 214 forecasts per ETF for GARCH and each of 4 GARCH-MIDAS drivers.
4494/4494 fits passed the convergence check.


## Choice of the GARCH-MIDAS driver

Forecast losses of the four GARCH-MIDAS candidates, with GARCH(1,1) as the benchmark, by ETF and pooled over all ETF-months (lower is better). MSE on log RV is $[\log(RV)-\log(F)]^2$; QLIKE is $RV/F-\log(RV/F)-1$.

In [8]:
ROW_NAMES = {"garch": "GARCH(1,1)", **{midas_label(d): d for d in DRIVERS}}
parts = []
for lab, name in ROW_NAMES.items():
    f = FC[lab]
    ratio = np.array([rv2.loc[t, tk] for t, tk in zip(f.target, f.etf)]) / f.rv2_hat.to_numpy()
    parts.append(pd.DataFrame({"model": name, "etf": f.etf.to_numpy(),
                               "MSE_log": np.log(ratio) ** 2, "QLIKE": ratio - np.log(ratio) - 1}))
cand = pd.concat(parts, ignore_index=True)

for loss in ["MSE_log", "QLIKE"]:
    table = cand.pivot_table(index="model", columns="etf", values=loss, aggfunc="mean")
    table["pooled"] = cand.groupby("model")[loss].mean()
    print(loss)
    display(table.reindex(index=list(ROW_NAMES.values()), columns=TICKERS + ["pooled"]).round(4))

# Preserve the complete candidate comparison for report Appendix B1.
candidate_per_etf = cand.groupby(["model", "etf"])[["MSE_log", "QLIKE"]].mean().reset_index()
candidate_pooled = cand.groupby("model")[["MSE_log", "QLIKE"]].mean().reset_index().assign(etf="pooled")
candidate_losses = pd.concat([candidate_per_etf, candidate_pooled], ignore_index=True)
FORECAST_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
candidate_losses.to_csv(FORECAST_RESULTS_DIR / "garch_midas_candidate_losses.csv", index=False)


MSE_log


etf,XLI,XLB,XLE,XLY,XLF,pooled
model,,,,,,
"GARCH(1,1)",0.4577,0.4347,0.4228,0.4760,0.5766,0.4736
UNRATE,0.4514,0.4262,0.4368,0.4714,0.5673,0.4706
CPIAUCSL,0.4611,0.4379,0.4254,0.4824,0.5779,0.4769
DGS3MO,0.4636,0.4409,0.4292,0.4838,0.5740,0.4783
BAA10Y,0.4734,0.4410,0.4336,0.4850,0.5800,0.4826


QLIKE


etf,XLI,XLB,XLE,XLY,XLF,pooled
model,,,,,,
"GARCH(1,1)",0.2607,0.2413,0.2552,0.2543,0.3061,0.2635
UNRATE,0.2653,0.2420,0.2665,0.2598,0.3121,0.2691
CPIAUCSL,0.2619,0.2459,0.2589,0.2638,0.3074,0.2676
DGS3MO,0.2626,0.2424,0.2623,0.2560,0.3079,0.2662
BAA10Y,0.2677,0.2383,0.2640,0.2590,0.3121,0.2682


**Why BAA10Y.**

* **No candidate dominates.** On QLIKE, all four have a higher pooled loss than GARCH(1,1), and the spread across them is small (0.2662 to 0.2691, about 1%); BAA10Y ranks third, behind DGS3MO and CPIAUCSL. On MSE on log RV, UNRATE has the lowest pooled loss, below GARCH's, and BAA10Y the highest.
* **QLIKE is the main loss for comparing the candidates.** Both models forecast the variance level, and QLIKE ranks such forecasts correctly even though realised variance measures true variance with error (Patton, 2011). MSE on log RV favours forecasts that are lower on average, because the log of an unbiased variance forecast overstates the expected log RV (Jensen's inequality). UNRATE's advantage on MSE on log RV does not carry over to QLIKE, where it is above GARCH for every ETF.
* **BAA10Y is the only candidate with a lower QLIKE than GARCH for any ETF** (XLB), although the gain is small.
* **Economic channel.** The credit spread is the candidate most directly tied to firms' default risk and financing conditions (Gilchrist and Zakrajšek, 2012), the channel through which macroeconomic conditions are expected to raise equity volatility.

BAA10Y is therefore the main specification: it is the only candidate that beats GARCH on QLIKE for at least one ETF (XLB), and it has the clearest economic link to equity volatility. The formal test of the chosen specification in `e1_garch_family.ipynb` is Clark-West on MSE on log RV, because Clark-West is defined for squared-error loss. The choice uses the same 2008–2025 forecasts that are evaluated, so the BAA10Y results are not an independent test of the driver.

## Export forecasts

Create a tidy forecast–outcome file for GARCH and BAA10Y GARCH-MIDAS, used by the separate evaluation notebook, plus a companion file with the BAA10Y `theta` at each origin. The file includes the forecast origin, target date, realised outcome and forecasts on both the log and variance scales.

`date` is the forecast-origin month-end; `target_date`, `actual_RV` and `actual_logRV` refer to the following month. Clark-West and Diebold-Mariano tests are in `e1_garch_family.ipynb`.


In [9]:
TIDY_COLS = ["model", "etf", "date", "target_date", "actual_logRV", "actual_RV", "fc_log", "sigma2", "fc_level"]


def to_tidy(labels):
    parts = []
    for lab in labels:
        f = FC[lab].copy()
        f["date"] = [o.to_timestamp(how="end").strftime("%Y-%m-%d") for o in f.origin]
        f["target_date"] = [t.to_timestamp(how="end").strftime("%Y-%m-%d") for t in f.target]
        f["actual_RV"] = [float(rv2.loc[t, tk]) for t, tk in zip(f.target, f.etf)]
        f["actual_logRV"] = np.log(f["actual_RV"])
        f["fc_level"] = f["rv2_hat"]
        f["fc_log"] = np.log(f["fc_level"])
        f["sigma2"] = np.nan  # GARCH forecasts variance directly on the level scale.
        f["etf"] = pd.Categorical(f["etf"], TICKERS, ordered=True)
        parts.append(f.sort_values(["etf", "origin"]))
    t = pd.concat(parts, ignore_index=True)[TIDY_COLS]
    t["etf"] = t["etf"].astype(str)
    return t


tidy = to_tidy(MODELS)
expected_rows = len(MODELS) * len(TICKERS) * len(TARGETS)
assert len(tidy) == expected_rows
assert not tidy.duplicated(["model", "etf", "date"]).any(), "Duplicate forecast keys"
assert set(tidy.model) == set(MODELS) and set(tidy.etf) == set(TICKERS)
assert tidy[["actual_RV", "actual_logRV", "fc_level", "fc_log"]].notna().all().all()
assert (tidy[["actual_RV", "fc_level"]] > 0).all().all()
assert (tidy["target_date"] > tidy["date"]).all(), "Target must follow its forecast origin"

# The realised target must be identical across the two models for a given ETF/origin.
actual_check = tidy.pivot(index=["etf", "date"], columns="model", values="actual_RV")
assert (actual_check.nunique(axis=1) == 1).all(), "Models disagree on realised targets"

tidy.to_csv(OUT_TIDY, index=False)

theta_out = FC[MIDAS_LABEL].copy()
theta_out["date"] = [o.to_timestamp(how="end").strftime("%Y-%m-%d") for o in theta_out.origin]
theta_out["target_date"] = [t.to_timestamp(how="end").strftime("%Y-%m-%d") for t in theta_out.target]
theta_out = theta_out[["model", "etf", "date", "target_date", "theta", "params_from"]].sort_values(["etf", "date"])
assert len(theta_out) == len(TICKERS) * len(TARGETS)
theta_out.to_csv(OUT_THETA, index=False)

# Round-trip check catches column-order and write issues before the evaluation notebook reads the file.
written = pd.read_csv(OUT_TIDY)
assert list(written.columns) == TIDY_COLS and len(written) == len(tidy)
assert np.allclose(written["fc_level"], tidy["fc_level"].to_numpy(), rtol=1e-12)

print(f"Saved {len(tidy)} rows to {OUT_TIDY} and {len(theta_out)} theta rows to {OUT_THETA}.")
display(tidy.head())


Saved 2140 rows to ./models/forecasting/forecasts/garch_forecasts.csv and 1070 theta rows to ./models/forecasting/forecasts/garchmidas_theta_by_origin.csv.


,model,etf,date,target_date,actual_logRV,actual_RV,fc_log,sigma2,fc_level
0,garch,XLI,2008-02-29,2008-03-31,-4.808556,0.008160,-5.657044,NaN,0.003493
1,garch,XLI,2008-03-31,2008-04-30,-5.344238,0.004776,-5.018257,NaN,0.006616
2,garch,XLI,2008-04-30,2008-05-31,-6.324658,0.001792,-5.315111,NaN,0.004917
3,garch,XLI,2008-05-31,2008-06-30,-5.424881,0.004406,-5.927957,NaN,0.002664
4,garch,XLI,2008-06-30,2008-07-31,-4.750961,0.008643,-5.447835,NaN,0.004306
